In [1]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 38.2 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import time
from datetime import datetime


if 'inventory' not in st.session_state:
    st.session_state.inventory = {
        "EQ001": {"name": "Router Cisco", "stock": 2, "total": 2},
        "EQ002": {"name": "Oscilloscope", "stock": 1, "total": 2},
        "EQ003": {"name": "Arduino Board", "stock": 5, "total": 5},
        "EQ004": {"name": "Multimeter", "stock": 0, "total": 3}
    }

if 'cart_stack' not in st.session_state:
    st.session_state.cart_stack = []

if 'waitlist_queue' not in st.session_state:
    st.session_state.waitlist_queue = {}

if 'system_logs' not in st.session_state:
    st.session_state.system_logs = []

def add_log(action, detail):
    timestamp = datetime.now().strftime("%H:%M:%S")
    st.session_state.system_logs.insert(0, f"[{timestamp}] {action}: {detail}")
    if len(st.session_state.system_logs) > 50:
        st.session_state.system_logs.pop()

class BSTNode:
    def __init__(self, booking_id, equip_id, student_name, timestamp):
        self.booking_id = booking_id
        self.equip_id = equip_id
        self.student_name = student_name
        self.timestamp = timestamp
        self.left = None
        self.right = None

class BookingBST:
    def __init__(self):
        self.root = None
        self.size = 0

    def insert(self, booking_id, equip_id, student_name, timestamp):
        if not self.root:
            self.root = BSTNode(booking_id, equip_id, student_name, timestamp)
        else:
            self._insert_recursive(self.root, booking_id, equip_id, student_name, timestamp)
        self.size += 1

    def _insert_recursive(self, node, booking_id, equip_id, student_name, timestamp):
        if booking_id < node.booking_id:
            if node.left is None: node.left = BSTNode(booking_id, equip_id, student_name, timestamp)
            else: self._insert_recursive(node.left, booking_id, equip_id, student_name, timestamp)
        elif booking_id > node.booking_id:
            if node.right is None: node.right = BSTNode(booking_id, equip_id, student_name, timestamp)
            else: self._insert_recursive(node.right, booking_id, equip_id, student_name, timestamp)

    def search(self, booking_id):
        return self._search_recursive(self.root, booking_id)

    def _search_recursive(self, node, booking_id):
        if node is None or node.booking_id == booking_id: return node
        if booking_id < node.booking_id: return self._search_recursive(node.left, booking_id)
        return self._search_recursive(node.right, booking_id)

    def get_all_bookings(self, inventory_ref):
        result = []
        self._inorder(self.root, result, inventory_ref)
        return result

    def _inorder(self, node, result, inventory_ref):
        if node:
            self._inorder(node.left, result, inventory_ref)
            eq_name = inventory_ref.get(node.equip_id, {}).get("name", "Unknown")
            result.append({
                "Booking ID": node.booking_id,
                "Time": node.timestamp,
                "Student": node.student_name,
                "Equipment": eq_name
            })
            self._inorder(node.right, result, inventory_ref)

    def render_tree(self):
        def _build_tree_string(node, prefix="", is_left=True):
            if not node: return ""
            res = ""
            if node.right:
                res += _build_tree_string(node.right, prefix + ("│   " if is_left else "    "), False)
            res += prefix + ("└── " if is_left else "┌── ") + str(node.booking_id) + "\n"
            if node.left:
                res += _build_tree_string(node.left, prefix + ("    " if is_left else "│   "), True)
            return res
        return _build_tree_string(self.root) if self.root else "No records"

if 'booking_bst' not in st.session_state:
    st.session_state.booking_bst = BookingBST()
if 'booking_counter' not in st.session_state:
    st.session_state.booking_counter = 5000

def quick_sort(arr, sort_by="name"):
    if len(arr) <= 1: return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x[sort_by] < pivot[sort_by]]
    middle = [x for x in arr if x[sort_by] == pivot[sort_by]]
    right = [x for x in arr if x[sort_by] > pivot[sort_by]]
    return quick_sort(left, sort_by) + middle + quick_sort(right, sort_by)


st.set_page_config(page_title="E-Lab Manager", page_icon="💠", layout="wide", initial_sidebar_state="expanded")


st.markdown("""
<style>
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
    header {visibility: hidden;}

    .glass-card {
        background: rgba(255, 255, 255, 0.05);
        border-radius: 12px;
        padding: 20px;
        border: 1px solid rgba(0, 0, 0, 0.1);
        box-shadow: 0 4px 6px rgba(0,0,0,0.05);
        margin-bottom: 20px;
    }
    div[data-testid="metric-container"] {
        background-color: #ffffff; border-left: 5px solid #4F8BF9; padding: 15px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);
    }
    .stButton>button { border-radius: 6px; font-weight: bold; transition: 0.2s; }
    .stButton>button:hover { transform: scale(1.02); }
</style>
""", unsafe_allow_html=True)


with st.sidebar:
    st.image("https://cdn-icons-png.flaticon.com/512/2004/2004240.png", width=60)
    st.markdown("## 💠 E-Lab System")
    st.caption("v1.0.0 Enterprise")
    st.markdown("---")
    menu = st.radio("NAVIGATION",
        ["🏠 1. Dashboard & Monitor",
         "📦 2. Inventory Management",
         "🛒 3. Cart Checkout",
         "⏳ 4. Queue Waitlist",
         "📊 5. System Reports",
         "📖 6. Transaction History"])

    st.markdown("---")
    st.markdown("### 💾 System State")
    st.progress(len(st.session_state.inventory) / 20, text=f"Registered Items: {len(st.session_state.inventory)}")
    st.progress(len(st.session_state.cart_stack) / 10, text=f"Items in Cart: {len(st.session_state.cart_stack)}")
    q_len = sum(len(q) for q in st.session_state.waitlist_queue.values())
    st.progress(q_len / 15, text=f"Users in Queue: {q_len}")
    st.progress(st.session_state.booking_bst.size / 50, text=f"Total Transactions: {st.session_state.booking_bst.size}")


if menu == "🏠 1. Dashboard & Monitor":
    st.title("🏠 System Dashboard")

    col_stat1, col_stat2, col_stat3, col_stat4 = st.columns(4)
    total_equip = sum([v["total"] for v in st.session_state.inventory.values()])
    col_stat1.metric("📦 Total Assets", total_equip)
    col_stat2.metric("🟢 Available Stock", sum([v["stock"] for v in st.session_state.inventory.values()]))
    col_stat3.metric("🔴 Checked Out", total_equip - sum([v["stock"] for v in st.session_state.inventory.values()]))
    col_stat4.metric("📊 Total Transactions", st.session_state.booking_bst.size)

    st.markdown("---")
    c_arch, c_log = st.columns([1.5, 1])

    with c_arch:
        st.markdown("<div class='glass-card'>", unsafe_allow_html=True)
        st.subheader("📊 Resource Utilization Analytics")
        st.caption("Real-time equipment allocation overview")

        borrowed_data = []
        for k, v in st.session_state.inventory.items():
            borrowed_amt = v["total"] - v["stock"]
            if borrowed_amt > 0:
                borrowed_data.append({"Item": v["name"], "Borrowed": borrowed_amt, "Total": v["total"]})

        if borrowed_data:
            df_borrowed = pd.DataFrame(borrowed_data).sort_values(by="Borrowed", ascending=False)
            max_val = max([d["Total"] for d in borrowed_data])
            st.dataframe(
                df_borrowed,
                column_config={
                    "Borrowed": st.column_config.ProgressColumn(
                        "Utilization (การใช้งาน)",
                        format="%d",
                        min_value=0,
                        max_value=max_val
                    ),
                    "Total": st.column_config.NumberColumn("Total Capacity")
                },
                use_container_width=True,
                hide_index=True
            )
        else:
            st.success("✅ All resources are currently available in the inventory. (ยังไม่มีการยืมอุปกรณ์)")

        st.markdown("---")
        st.subheader("⚡ System Diagnostics")
        col_diag1, col_diag2 = st.columns(2)
        col_diag1.metric("Server Status", "Online", "Stable")
        col_diag2.metric("Sync Latency", "12 ms", "-2 ms", delta_color="inverse")
        st.markdown("</div>", unsafe_allow_html=True)

    with c_log:
        st.subheader("📡 Live System Logs")
        with st.container(height=380, border=True):
            if st.session_state.system_logs:
                for log in st.session_state.system_logs:
                    if "ADD" in log or "JOIN" in log or "SAVE" in log: st.markdown(f"🟢 `{log}`")
                    elif "REMOVE" in log or "ASSIGN" in log or "RETURN" in log or "DELETE" in log: st.markdown(f"🟠 `{log}`")
                    else: st.markdown(f"⚪ `{log}`")
            else: st.caption("No system activity yet.")


elif menu == "📦 2. Inventory Management":
    st.title("📦 Inventory Management")

    t_view, t_add, t_del = st.tabs(["📋 View Data", "➕ Add/Update", "🗑️ Delete Equipment"])

    with t_view:
        df = pd.DataFrame([{"ID": k, "Name": v["name"], "Stock": v["stock"], "Total": v["total"]} for k, v in st.session_state.inventory.items()])
        st.dataframe(df, use_container_width=True, hide_index=True)

    with t_add:
        with st.form("add_form", clear_on_submit=True):
            c1, c2, c3 = st.columns([1, 2, 1])
            eq_id = c1.text_input("🔑 Equipment ID")
            eq_name = c2.text_input("📌 Name")
            eq_qty = c3.number_input("📦 Total Quantity", min_value=1)

            if st.form_submit_button("💾 Save Data", type="primary"):
                if eq_id and eq_name:
                    st.session_state.inventory[eq_id] = {"name": eq_name, "stock": eq_qty, "total": eq_qty}
                    add_log("SAVE", f"Updated {eq_name} ({eq_id})")
                    st.success("Saved successfully!")
                    time.sleep(0.5); st.rerun()
                else: st.error("Please fill all fields.")

    with t_del:
        del_options = [f"{k} - {v['name']}" for k, v in st.session_state.inventory.items()]
        to_delete = st.selectbox("Select Equipment to Delete:", del_options)
        if st.button("🗑️ Delete Data"):
            d_id = to_delete.split(" - ")[0]
            del st.session_state.inventory[d_id]
            add_log("DELETE", f"Removed item {d_id}")
            st.warning(f"Deleted {d_id} successfully!")
            time.sleep(0.5); st.rerun()


elif menu == "🛒 3. Cart Checkout":
    st.title("🛒 Cart Checkout")

    col_l, col_r = st.columns([1.5, 1])
    with col_l:
        st.markdown("<div class='glass-card'>", unsafe_allow_html=True)
        student_name = st.text_input("👤 Student Name")
        opts = [f"{k} - {v['name']} (Stock: {v['stock']})" for k, v in st.session_state.inventory.items()]
        selected = st.selectbox("Select Item:", opts)

        if st.button("📥 Add to Cart", type="primary", use_container_width=True):
            eq_id = selected.split(" - ")[0]
            if st.session_state.inventory[eq_id]["stock"] > 0:
                st.session_state.cart_stack.append(eq_id)
                st.session_state.inventory[eq_id]["stock"] -= 1
                add_log("ADD", f"Added {eq_id} to Cart")
                st.rerun()
            else: st.error("Out of stock! Please join the waitlist.")
        st.markdown("</div>", unsafe_allow_html=True)

    with col_r:
        st.subheader("🛒 Current Cart")
        with st.container(border=True, height=220):
            if st.session_state.cart_stack:
                for i, eid in enumerate(reversed(st.session_state.cart_stack)):
                    if i == 0: st.markdown(f"**👉 Latest:** `{st.session_state.inventory[eid]['name']}`")
                    else: st.markdown(f"**⬇️** `{st.session_state.inventory[eid]['name']}`")
            else: st.caption("Cart is empty")

        c_pop, c_chk = st.columns(2)
        if c_pop.button("📤 Undo (Remove Last)", use_container_width=True) and st.session_state.cart_stack:
            rm_id = st.session_state.cart_stack.pop()
            st.session_state.inventory[rm_id]["stock"] += 1
            add_log("REMOVE", f"Removed {rm_id} from Cart")
            st.rerun()

        if c_chk.button("✅ CHECKOUT", type="primary", use_container_width=True) and st.session_state.cart_stack:
            if student_name:
                curr_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                for eid in st.session_state.cart_stack:
                    b_id = st.session_state.booking_counter
                    st.session_state.booking_bst.insert(b_id, eid, student_name, curr_time)
                    add_log("SAVE", f"Booked {b_id} for {student_name}")
                    st.session_state.booking_counter += 1
                st.session_state.cart_stack.clear()
                st.success("Transaction Complete!"); st.balloons()
            else: st.error("Enter Student Name!")


elif menu == "⏳ 4. Queue Waitlist":
    st.title("⏳ Queue Waitlist")

    t_enq, t_deq = st.tabs(["📥 Join Waitlist", "📤 Return & Reassign"])

    with t_enq:
        w_user = st.text_input("🙋‍♂️ Student Name")
        out_items = [f"{k} - {v['name']}" for k, v in st.session_state.inventory.items() if v['stock'] == 0]
        if out_items:
            w_eq = st.selectbox("Select Item:", out_items)
            if st.button("Join Waitlist", type="primary"):
                if w_user:
                    w_id = w_eq.split(" - ")[0]
                    if w_id not in st.session_state.waitlist_queue: st.session_state.waitlist_queue[w_id] = []
                    st.session_state.waitlist_queue[w_id].append(w_user)
                    add_log("JOIN", f"{w_user} joined waitlist for {w_id}")
                    st.success("Added to Waitlist!"); time.sleep(0.5); st.rerun()
        else: st.success("All items have stock available.")

    with t_deq:
        borrowed = [f"{k} - {v['name']}" for k, v in st.session_state.inventory.items() if v['stock'] < v['total']]
        if borrowed:
            ret_eq = st.selectbox("Select Item to Return:", borrowed)
            if st.button("Return Item"):
                r_id = ret_eq.split(" - ")[0]
                curr_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                if r_id in st.session_state.waitlist_queue and st.session_state.waitlist_queue[r_id]:
                    next_user = st.session_state.waitlist_queue[r_id].pop(0)
                    b_id = st.session_state.booking_counter
                    st.session_state.booking_bst.insert(b_id, r_id, next_user, curr_time)
                    st.session_state.booking_counter += 1
                    add_log("ASSIGN", f"Reassigned {r_id} to {next_user}")
                    st.success(f"Item returned and instantly assigned to **{next_user}**!")
                else:
                    st.session_state.inventory[r_id]["stock"] += 1
                    add_log("RETURN", f"Returned {r_id} to inventory")
                    st.success("Item returned to inventory.")
        else: st.info("No items are currently borrowed.")

    st.markdown("---")
    st.subheader("🚦 Waitlist Status")
    for eq, queue in st.session_state.waitlist_queue.items():
        if queue:
            st.markdown(f"**{st.session_state.inventory[eq]['name']}**")
            q_vis = " ➡️ ".join([f"[{i+1}] {n}" for i, n in enumerate(queue)])
            st.info(f"🟢 Next in line -> {q_vis} -> Last 🔴")


elif menu == "📊 5. System Reports":
    st.title("📊 System Reports")

    sort_by = st.radio("Sort Criteria:", ["🔤 Name (A-Z)", "🔢 Stock (Low to High)"], horizontal=True)

    eq_list = [{"id": k, "name": v["name"], "stock": v["stock"], "total": v["total"]} for k, v in st.session_state.inventory.items()]

    with st.spinner("Processing report..."):
        time.sleep(0.3)
        sorted_list = quick_sort(eq_list, "name" if "A-Z" in sort_by else "stock")
        df_sort = pd.DataFrame(sorted_list)
        st.dataframe(df_sort, use_container_width=True)
        st.download_button("📥 Download Report (CSV)", df_sort.to_csv(index=False).encode('utf-8'), 'system_report.csv', 'text/csv')


elif menu == "📖 6. Transaction History":
    st.title("📖 Transaction History")

    t_data, t_search, t_viz = st.tabs(["🗂️ All Records", "🔍 Search Record", "🌳 Data Hierarchy"])

    with t_data:
        all_books = st.session_state.booking_bst.get_all_bookings(st.session_state.inventory)
        if all_books: st.dataframe(pd.DataFrame(all_books), use_container_width=True)
        else: st.info("No transactions yet.")

    with t_search:
        with st.container(border=True):
            sid = st.number_input("Enter Booking ID to search:", min_value=5000, value=5000)
            if st.button("Search", type="primary"):
                res = st.session_state.booking_bst.search(sid)
                if res:
                    st.success(f"**Record Found!**\n\nStudent: `{res.student_name}` | Equipment: `{st.session_state.inventory.get(res.equip_id, {}).get('name', 'N/A')}` | Time: `{res.timestamp}`")
                    add_log("SEARCH", f"Found ID {sid}")
                else:
                    st.error("Booking ID not found.")
                    add_log("SEARCH", f"ID {sid} not found")

    with t_viz:
        st.subheader("🌳 Data Hierarchy")
        st.caption("Visual representation of the transaction record links")
        tree_string = st.session_state.booking_bst.render_tree()
        st.code(tree_string, language="bash")

Writing app.py


In [3]:
from pyngrok import ngrok
import os

NGROK_AUTH_TOKEN = "36GtyDmeo0zoekJRRrkvHjTnYEB_3ivhw6zZcu5CKS1YnzU4H"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

ngrok.kill()

os.system("streamlit run app.py &")

public_url = ngrok.connect(8501).public_url

print("-" * 50)
print(f"✅ เซิร์ฟเวอร์พร้อมใช้งานแล้ว!")
print(f"🌍 คลิกลิงก์นี้เพื่อเปิดหน้าระบบ: {public_url}")
print("-" * 50)

--------------------------------------------------
✅ เซิร์ฟเวอร์พร้อมใช้งานแล้ว!
🌍 คลิกลิงก์นี้เพื่อเปิดหน้าระบบ: https://multispermous-otilia-unempoisoned.ngrok-free.dev
--------------------------------------------------
